# Axon Compression Analysis

This notebook analyses trajectory compression results from the Axon SDK.

**Usage:**
1. Run the synthetic benchmark first: `python examples/benchmark/run_benchmark.py --output benchmark_results.json`
2. Set `RESULTS_FILE` to point at the output JSON, or point at the community benchmark registry API.
3. Execute all cells.

> **Note on data:** All `[TBD]` placeholders in this notebook will be
> replaced with real production data once it is available from the community
> benchmark registry.  No numbers are fabricated.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
from __future__ import annotations

import json
import pathlib
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print('Imports OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────

# Path to benchmark results JSON produced by run_benchmark.py
RESULTS_FILE = pathlib.Path('../../examples/benchmark/benchmark_results.json')

# Community benchmark registry API (set BACKEND_URL to your deployment)
BACKEND_URL = 'http://localhost:8000'
BENCHMARKS_ENDPOINT = f'{BACKEND_URL}/v1/benchmarks'

print(f'Results file: {RESULTS_FILE}')
print(f'Benchmarks endpoint: {BENCHMARKS_ENDPOINT}')

In [ ]:
# ── Load local benchmark results (synthetic) ───────────────────────────────

def load_local_results(path: pathlib.Path) -> pd.DataFrame:
    """Load benchmark results from a local JSON file."""
    if not path.exists():
        print(f'File not found: {path}. Run run_benchmark.py first.')
        return pd.DataFrame()
    with path.open() as f:
        data: list[dict[str, Any]] = json.load(f)
    return pd.DataFrame(data)

df_local = load_local_results(RESULTS_FILE)
print(f'Loaded {len(df_local)} local benchmark rows')
if not df_local.empty:
    display(df_local.head())

In [ ]:
# ── Load community benchmark data from the registry ────────────────────────

import urllib.request
import urllib.error

def load_registry_data(endpoint: str, limit: int = 500) -> pd.DataFrame:
    """Fetch community benchmark submissions from the public registry."""
    url = f'{endpoint}?limit={limit}'
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:  # noqa: S310
            payload: list[dict[str, Any]] = json.loads(resp.read())
        return pd.DataFrame(payload)
    except urllib.error.URLError as exc:
        print(f'Could not reach registry ({exc}). Is the backend running?')
        return pd.DataFrame()

df_registry = load_registry_data(BENCHMARKS_ENDPOINT)
print(f'Loaded {len(df_registry)} registry rows')
if not df_registry.empty:
    display(df_registry.describe())

In [ ]:
# ── Compression ratio distribution ─────────────────────────────────────────
# Requires df_registry to be non-empty.

def plot_compression_distribution(df: pd.DataFrame) -> None:
    """Plot p50 and p95 compression ratio distributions."""
    if df.empty:
        print('No data to plot. Load registry data first.')
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df['p50_compression_ratio'].dropna(), bins=20, color='teal', edgecolor='white')
    axes[0].set_title('p50 Compression Ratio Distribution')
    axes[0].set_xlabel('Compression Ratio')
    axes[0].set_ylabel('Count')

    axes[1].hist(df['p95_compression_ratio'].dropna(), bins=20, color='steelblue', edgecolor='white')
    axes[1].set_title('p95 Compression Ratio Distribution')
    axes[1].set_xlabel('Compression Ratio')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.savefig('compression_ratio_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: compression_ratio_distribution.png')

plot_compression_distribution(df_registry)

In [ ]:
# ── Cost distribution ──────────────────────────────────────────────────────

def plot_cost_distribution(df: pd.DataFrame) -> None:
    """Plot p50 and p95 cost distributions from registry submissions."""
    if df.empty:
        print('No data to plot.')
        return

    try:
        p50 = df['p50_cost_usd'].astype(float)
        p95 = df['p95_cost_usd'].astype(float)
    except (KeyError, ValueError) as exc:
        print(f'Could not parse cost columns: {exc}')
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(p50.dropna(), bins=20, color='coral', edgecolor='white')
    axes[0].set_title('p50 Cost per Call (USD)')
    axes[0].set_xlabel('Cost (USD)')
    axes[0].set_ylabel('Count')

    axes[1].hist(p95.dropna(), bins=20, color='salmon', edgecolor='white')
    axes[1].set_title('p95 Cost per Call (USD)')
    axes[1].set_xlabel('Cost (USD)')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.savefig('cost_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: cost_distribution.png')

plot_cost_distribution(df_registry)

In [ ]:
# ── Routing accuracy analysis ──────────────────────────────────────────────

def routing_accuracy_summary(df: pd.DataFrame) -> None:
    """Print summary statistics for avg_routing_accuracy."""
    if df.empty or 'avg_routing_accuracy' not in df.columns:
        print('No routing accuracy data available.')
        return

    acc = df['avg_routing_accuracy'].dropna()
    print('Routing Accuracy Summary (community submissions)')
    print(f'  Count : {len(acc)}')
    print(f'  Mean  : {acc.mean():.4f}')
    print(f'  Median: {acc.median():.4f}')
    print(f'  p5    : {np.percentile(acc, 5):.4f}')
    print(f'  p95   : {np.percentile(acc, 95):.4f}')
    print(f'  Min   : {acc.min():.4f}')
    print(f'  Max   : {acc.max():.4f}')

routing_accuracy_summary(df_registry)

In [ ]:
# ── SDK version adoption ───────────────────────────────────────────────────

def plot_sdk_version_adoption(df: pd.DataFrame) -> None:
    """Bar chart of submission counts by SDK version."""
    if df.empty or 'sdk_version' not in df.columns:
        print('No SDK version data available.')
        return

    counts = df['sdk_version'].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(8, 4))
    counts.plot(kind='bar', ax=ax, color='teal', edgecolor='white')
    ax.set_title('Benchmark Submissions by SDK Version')
    ax.set_xlabel('SDK Version')
    ax.set_ylabel('Submission Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('sdk_version_adoption.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: sdk_version_adoption.png')

plot_sdk_version_adoption(df_registry)

In [ ]:
# ── Summary table for paper ────────────────────────────────────────────────

def generate_paper_summary(df: pd.DataFrame) -> None:
    """Print a Markdown summary table suitable for inserting into the paper."""
    if df.empty:
        print('No data available for paper summary.')
        return

    try:
        p50_cr = df['p50_compression_ratio'].dropna()
        p95_cr = df['p95_compression_ratio'].dropna()
        acc = df['avg_routing_accuracy'].dropna()
    except KeyError as exc:
        print(f'Missing column: {exc}')
        return

    print('## Community Benchmark Summary (for paper Section 5.4)')
    print()
    print(f'N = {len(df)} submissions')
    print()
    print('| Metric | Mean | Median | p5 | p95 |')
    print('|---|---|---|---|---|')
    print(
        f'| p50 compression ratio | {p50_cr.mean():.3f} | {p50_cr.median():.3f} |'
        f' {np.percentile(p50_cr, 5):.3f} | {np.percentile(p50_cr, 95):.3f} |'
    )
    print(
        f'| p95 compression ratio | {p95_cr.mean():.3f} | {p95_cr.median():.3f} |'
        f' {np.percentile(p95_cr, 5):.3f} | {np.percentile(p95_cr, 95):.3f} |'
    )
    print(
        f'| avg routing accuracy  | {acc.mean():.3f}  | {acc.median():.3f}  |'
        f' {np.percentile(acc, 5):.3f}  | {np.percentile(acc, 95):.3f}  |'
    )

generate_paper_summary(df_registry)